# Phase 7 prep run analysis (2026-04-20)

First clean end-to-end run on the Commit-G manifest-as-oracle architecture (schema v2, per-hull applicability, built-in-aware by construction).

**Config**: 8 hulls × `early` regime × seed 0 × sim_budget=600 · TPE · 96 c7a.2xlarge spot across us-east-1 + us-east-2.

**Inputs**: `data/logs/<hull>__early__tpe__seed0/evaluation_log.jsonl` (per-trial records) · `~/starsector-campaigns/phase7-prep-early/ledger.jsonl` (per-worker heartbeat cost rows).

**Fitness pipeline** (5A–5E stack):
```
per-matchup CombatResult
   ↓ aggregate_combat_fitness(mode='mean')
raw_fitness  (per-trial scalar)
   ↓ TWFE decomposition (5A): subtract opponent FE
twfe_fitness = α̂  (build effect, opponent-mix-neutralized)
   ↓ EB shrinkage toward regression prior on 10-dim covariates (5D)
eb_fitness = w·α̂ + (1-w)·γ̂ᵀ[1,X]
   ↓ Box-Cox warp + min-max rescale (5E)
fitness = shaped  ∈ [0, 1]   ← what TPE sees
```
Each transform ADDS information to the *sampler's* view without changing the ranking. For **analyst** comparisons across time, refit once on the final population and work in α̂ space (Box-Cox is monotone → preserves ordering but compresses scale; min-max forces current-best to 1.0 by construction).

In [ ]:
import json, glob, os, sys, warnings
from collections import Counter, defaultdict
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt

sys.path.insert(0, '../src')
from starsector_optimizer.deconfounding import eb_shrinkage
from starsector_optimizer.models import EBShrinkageConfig
warnings.filterwarnings('ignore', category=UserWarning)

LOG_GLOB = '../data/logs/*__early__tpe__seed0'
LEDGER   = Path.home() / 'starsector-campaigns/phase7-prep-early/ledger.jsonl'

## 1. Per-study summary

In [ ]:
def load_study(path):
    rows = []
    for line in open(path):
        try: rows.append(json.loads(line))
        except json.JSONDecodeError: pass
    return rows

studies = {}
for d in sorted(glob.glob(LOG_GLOB)):
    hull = os.path.basename(d).split('__')[0]
    studies[hull] = load_study(f'{d}/evaluation_log.jsonl')

summary = []
for hull, rows in studies.items():
    finalized = [r for r in rows if not r.get('pruned')]
    pruned    = [r for r in rows if r.get('pruned')]
    eb_ok     = [r for r in finalized if r.get('eb_fitness') != r.get('twfe_fitness')]  # activated rows
    summary.append({
        'hull': hull,
        'total_rows': len(rows),
        'finalized':  len(finalized),
        'pruned':     len(pruned),
        'prune_rate': round(len(pruned) / max(len(rows), 1), 3),
        'eb_activated': len(eb_ok),
        'eb_activation_rate': round(len(eb_ok) / max(len(finalized), 1), 3),
        'raw_fitness_max':  round(max(r['raw_fitness'] for r in rows), 3),
        'twfe_max':         round(max(r.get('twfe_fitness', -9) for r in rows), 3),
    })
df_summary = pd.DataFrame(summary).set_index('hull')
df_summary

## 2. Cost attribution
Per-region, per-study breakdown of spot spend.

In [ ]:
ledger_rows = [json.loads(l) for l in open(LEDGER) if l.strip()]
heartbeats = [r for r in ledger_rows if r.get('event_type') == 'worker_heartbeat']
df_ledger = pd.DataFrame(heartbeats)
df_ledger['timestamp'] = pd.to_datetime(df_ledger['timestamp'])

print(f"Total heartbeat rows: {len(df_ledger):,}")
print(f"Cumulative cost:      ${df_ledger['cumulative_usd'].iloc[-1]:.2f}")
print(f"Unique workers:       {df_ledger['worker_id'].nunique()}")
print()
region_cost = df_ledger.groupby(['region', 'instance_type'])['delta_usd'].sum().round(2)
print('Cost by region × instance:')
print(region_cost)

In [ ]:
# Cost-over-time plot with warn threshold annotations
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_ledger['timestamp'], df_ledger['cumulative_usd'], lw=1)
for frac, color in [(0.5, 'gold'), (0.8, 'orange'), (0.95, 'red')]:
    ax.axhline(70 * frac, color=color, ls='--', alpha=0.5, label=f'{int(frac*100)}% of $70')
ax.set(xlabel='time (UTC)', ylabel='cumulative USD', title='Ledger cumulative cost vs warn thresholds')
ax.legend(); plt.xticks(rotation=20); plt.tight_layout(); plt.show()

## 3. Unified-fit running-best trajectory (the right way)

The `fitness` field stored per row was shaped under that trial's snapshot of the EB+Box-Cox fit. Those numbers are **not** comparable across time. To answer "is the optimizer still improving?", re-run EB on the full per-study population **once** and work in α̂ scale (skip Box-Cox + min-max — both are monotone, so the ranking is preserved but the scale becomes meaningless post-rescale).

In [ ]:
def refit_eb(rows, min_n=8):
    keep = [r for r in rows
            if r.get('twfe_fitness') is not None
            and r.get('covariate_vector')
            and not r.get('pruned')]
    if len(keep) < min_n: return None
    alpha  = np.array([r['twfe_fitness'] for r in keep], float)
    X      = np.array([r['covariate_vector'] for r in keep], float)
    # Prefer stored per-trial σ̂² if present; else fall back to variance/n.
    def sigma2_for(r):
        d = r.get('eb_diagnostics') or {}
        return d.get('sigma_sq_twfe') or float(np.var(alpha) / max(len(alpha)-1, 1))
    sigma2 = np.array([sigma2_for(r) for r in keep], float)
    eb, gamma, tau2, kept_cols = eb_shrinkage(alpha, sigma2, X, EBShrinkageConfig())
    trial_no = np.array([r['trial_number'] for r in keep])
    order = np.argsort(trial_no)
    return dict(alpha=alpha[order], eb=eb[order], gamma=gamma, tau2=tau2,
                kept_cols=kept_cols, trials=trial_no[order], n=len(keep))

fits = {h: refit_eb(rows) for h, rows in studies.items()}
for h, f in fits.items():
    if f: print(f'{h:11s} n={f["n"]:3d}  α̂ max={f["alpha"].max():+.3f}  EB max={f["eb"].max():+.3f}  τ̂²={f["tau2"]:.4f}  kept_cov={len(f["kept_cols"])}/10')

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7), sharey=False)
for ax, (hull, f) in zip(axes.ravel(), fits.items()):
    if f is None: continue
    running = np.maximum.accumulate(f['eb'])
    ax.plot(f['trials'], f['eb'], 'o', ms=1.5, alpha=0.35, label='EB α̂')
    ax.plot(f['trials'], running, '-', lw=1.5, color='red', label='running-best')
    ax.set_title(f'{hull} (n={f["n"]})'); ax.grid(alpha=0.3)
    ax.set_xlabel('trial #')
axes[0, 0].legend(loc='lower right', fontsize=8)
plt.suptitle('Per-study running-best under unified EB refit (α̂ scale)', y=1.02)
plt.tight_layout(); plt.show()

### Q4 breakout test
A *breakout* = running-best improved in final quartile by ≥ 10% of the final value. This validates the sim_budget=600 choice vs early truncation at 300–400 trials.

In [ ]:
breakouts = []
for hull, f in fits.items():
    if f is None: continue
    eb_ord = f['eb']; n = len(eb_ord)
    q3_end = 3 * n // 4
    best_q3 = eb_ord[:q3_end].max() if q3_end else -np.inf
    best_final = eb_ord.max()
    delta = best_final - best_q3
    rel = delta / abs(best_final) if abs(best_final) > 1e-6 else 0
    breakouts.append({
        'hull': hull, 'best_final_α̂': round(best_final, 3),
        'best_q3_α̂': round(best_q3, 3), 'Δ': round(delta, 3),
        'rel_improvement': round(rel, 3),
        'Q4_breakout (≥10%)': rel >= 0.10,
    })
pd.DataFrame(breakouts).set_index('hull').sort_values('Δ', ascending=False)

## 4. EB shrinkage diagnostics per hull

In [ ]:
# Covariate name order pinned in optimizer.py._build_covariate_vector
COV_NAMES = [
    'eff_max_flux', 'eff_flux_dissipation', 'eff_armor_rating',
    'eff_hull_hp_pct', 'ballistic_range_bonus', 'shield_damage_taken_mult',
    'total_weapon_dps', 'engagement_range', 'kinetic_dps_fraction',
    'op_used_fraction',
]

rows = []
for hull, f in fits.items():
    if f is None: continue
    # γ̂ has shape (1 + p_kept,) — intercept + kept covariate slopes
    gamma = f['gamma']
    kept = f['kept_cols']
    entry = {'hull': hull, 'τ̂²': round(f['tau2'], 4), 'γ̂_intercept': round(gamma[0], 3)}
    for name in COV_NAMES:
        entry[name] = np.nan
    for i, col in enumerate(kept):
        entry[COV_NAMES[col]] = round(gamma[1 + i], 3)
    rows.append(entry)
pd.DataFrame(rows).set_index('hull')

In [ ]:
# EB shrinkage weight distribution: w_i = τ̂² / (τ̂² + σ̂_i²)
# w close to 1 → trial's raw α̂ dominates; w close to 0 → prior dominates.
fig, ax = plt.subplots(figsize=(10, 5))
for hull, rows in studies.items():
    diags = [r.get('eb_diagnostics') for r in rows if r.get('eb_diagnostics')]
    if not diags: continue
    ws = [d['tau2'] / (d['tau2'] + d['sigma_sq_twfe'])
          for d in diags
          if d and d.get('tau2', 0) > 0 and d.get('sigma_sq_twfe', 0) > 0]
    if ws:
        ax.hist(ws, bins=30, alpha=0.4, label=f'{hull} (n={len(ws)})')
ax.set(xlabel='w_i = τ̂² / (τ̂² + σ̂_i²)', ylabel='trials', title='EB shrinkage weight distribution per hull')
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

## 5. ASHA pruning effectiveness

In [ ]:
prune_by_rung = defaultdict(Counter)
for hull, rows in studies.items():
    for r in rows:
        if r.get('pruned'):
            rung = r.get('opponents_evaluated') or 0
            prune_by_rung[hull][rung] += 1

df_rung = pd.DataFrame(prune_by_rung).T.fillna(0).astype(int)
df_rung = df_rung.reindex(columns=sorted(df_rung.columns))
df_rung.columns = [f'rung_{c}' for c in df_rung.columns]
df_rung['total_pruned'] = df_rung.sum(axis=1)
df_rung

In [ ]:
# Wasted matchups avoided: pruned trials would have run (10 - rung_pruned) more matchups each.
# N_opps_per_trial = 10 (hull-sized pool).
N_OPPS = 10
saved = []
for hull, rows in studies.items():
    saved_matchups = sum(N_OPPS - (r.get('opponents_evaluated') or 0)
                         for r in rows if r.get('pruned'))
    finalized_matchups = sum(r.get('opponents_evaluated') or 0
                             for r in rows if not r.get('pruned'))
    total_possible = len(rows) * N_OPPS
    saved.append({
        'hull': hull,
        'finalized_matchups': finalized_matchups,
        'saved_matchups': saved_matchups,
        'total_if_no_prune': total_possible,
        'compute_saved_pct': round(100 * saved_matchups / total_possible, 1) if total_possible else 0,
    })
pd.DataFrame(saved).set_index('hull')

## 6. Top-3 builds per hull

Ranked by EB-shrunken α̂ (refit on final data, not the per-trial stored `fitness`).

In [ ]:
for hull, rows in studies.items():
    f = fits[hull]
    if f is None: continue
    keep = [r for r in rows
            if r.get('twfe_fitness') is not None
            and r.get('covariate_vector')
            and not r.get('pruned')]
    keep_sorted = sorted(zip(keep, f['eb']), key=lambda kv: -kv[1])[:3]
    print(f'\n=== {hull.upper()} (n={f["n"]}) ===')
    for i, (r, ebv) in enumerate(keep_sorted, 1):
        b = r['build']
        weapons = {k: v for k, v in b.get('weapon_assignments', {}).items() if v}
        hullmods = sorted(b.get('hullmods', []))
        vents = b.get('flux_vents', 0); caps = b.get('flux_capacitors', 0)
        print(f'  #{i} trial={r["trial_number"]} EB_α̂={ebv:+.3f} raw_α̂={r["twfe_fitness"]:+.3f}')
        print(f'      Vents={vents} Caps={caps}')
        print(f'      Weapons ({len(weapons)}): {", ".join(sorted(weapons.values()))}')
        print(f'      Hullmods ({len(hullmods)}): {", ".join(hullmods)}')

## 7. Heuristic vs combat-simulated α̂ correlation

Was `composite_score` (the 11–22% of |γ̂| driver the plan dropped from covariates) a useful *heuristic* signal even if it's not admissible as a covariate?

In [ ]:
from scipy.stats import spearmanr
rows = []
for hull, study_rows in studies.items():
    f = fits[hull]
    if f is None: continue
    keep = [r for r in study_rows
            if r.get('twfe_fitness') is not None and not r.get('pruned') and r.get('covariate_vector')]
    if len(keep) < 30: continue
    # composite_score is no longer in the covariate_vector (post-Commit-D). But engagement_range
    # (index 7) is a raw scorer output we kept. Use it as the illustrative heuristic proxy.
    eng_range = np.array([r['covariate_vector'][7] for r in keep])
    alpha_hat = f['alpha']
    rho, p = spearmanr(eng_range, alpha_hat)
    rows.append({'hull': hull, 'ρ(engagement_range, α̂)': round(rho, 3), 'p': round(p, 3), 'n': len(keep)})
pd.DataFrame(rows).set_index('hull')

## 8. Pre-matchup covariate importance

γ̂ coefficient magnitudes per covariate, averaged across hulls. Tells us which engine-truth + Python-raw features are doing real work in the EB prior.

In [ ]:
gammas = {name: [] for name in COV_NAMES}
for hull, f in fits.items():
    if f is None: continue
    for i, col in enumerate(f['kept_cols']):
        gammas[COV_NAMES[col]].append(abs(f['gamma'][1 + i]))

importance = sorted(
    [(name, np.mean(vs) if vs else 0, np.std(vs) if vs else 0, len(vs))
     for name, vs in gammas.items()],
    key=lambda t: -t[1],
)
df_imp = pd.DataFrame(importance, columns=['covariate', 'mean|γ̂|', 'std', 'n_hulls_retained'])
df_imp.set_index('covariate')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df_imp_plot = df_imp.sort_values('mean|γ̂|')
ax.barh(df_imp_plot['covariate'], df_imp_plot['mean|γ̂|'], xerr=df_imp_plot['std'])
ax.set(xlabel='mean |γ̂| across 8 hulls', title='Covariate importance in EB regression prior')
plt.tight_layout(); plt.show()